# A3 移植 (3/8)：M2 —— AICPU 解算器

> **一句话结论**：A3 编译链不支持 SIMT，上游 resolver 算法被逐字转写成 AICPU C++；分两阶段跑，scratch 按 device 缓存复用，保证图捕获内无堆分配。

## 1. 问题是什么

A5 的 resolver 是 AscendC SIMT AIV 内核。A3 的编译目标 `dav-c220` 在 CANN 9.x 没有 `asc_simt` 支持，SIMT 编译直接失败，offload 根本起不来。

## 2. 转写方式

上游算法**逐字转写**为 AICPU C++（kernel 558 行），五步语义不变：fast path → hash union → hot 命中 → 稳定 LRU → victim 回填。

### 两阶段并行

- **Phase 1**：`CpuKernelUtils::ParallelFor` 按行分片处理 fast-path——只读共享表、写互斥切片，天然安全；这是 AICPU 6 核唯一的用武之地；
- **Phase 2**：串行处理共享 hash scratch（开放寻址 union）。

### 捕获安全

- scratch 按 device 缓存复用，不在图捕获内申请堆内存；
- 输出预分配；几何门控：topk=2048、block=128、hot>0 才走该路径。

## 3. 接口规模

20 个输入张量原地改写，25 个几何属性；输出物理索引、搬移清单、stats 与有效长度。

## 4. 成本（4k bs16）

- 图内 264 µs/次，每步 20.8 次；
- 合计 **+5.24 ms/步**，是 A3 账本里下一最大单项；
- 服务零错误，语义与上游逐位对齐；17 项 op 测试在 A3 通过。

## 5. 小结

- M2 = SIMT 不可用 → AICPU 逐字转写；
- Phase 1 行级并行，Phase 2 串行共享 hash；scratch 缓存复用；
- 代价 +5.24 ms/步；后续可考虑 AI-Core 化；
- 下一篇讲 M3：写槽失效从 Triton 搬到 AICPU。

**下一篇**：`04-m3-aicpu-invalidate.ipynb`
